# Day 4 - Joins and Unions in PySpark

This notebook covers combining DataFrames in PySpark:
- Reading CSV files
- Inner join
- Broadcast join
- Unions


## 1. Create SparkSession

`SparkSession` is the unified entry point for Spark operations.


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("payspark_day4")\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/06 15:36:07 WARN Utils: Your hostname, biswajits, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/06 15:36:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/biswa/practice/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/06 15:36:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Read CSV Files

Read both datasets needed for joining:
- `Customers.csv` - customer information
- `Orders.csv` - order information

Both are read with `header=True`. Without `inferSchema`, all columns are read as strings.


In [3]:
path_1 = r"../dataset/Customers.csv"
path_2 = r"../dataset/Orders.csv"

df_1 = spark.read.csv(path_1, header=True)
df_2 = spark.read.csv(path_2, header=True)
print(df_1)
print(df_2)

DataFrame[CustomerID: string, FirstName: string, LastName: string, Country: string, Score: string]
DataFrame[OrderID: string, ProductID: string, CustomerID: string, SalesPersonID: string, OrderDate: string, ShipDate: string, OrderStatus: string, ShipAddress: string, BillAddress: string, Quantity: string, Sales: string, CreationTime: string]


## 3. Joins

`join()` combines two DataFrames based on a common key column.

### 3.1 Inner Join

Inner join returns only the rows where the join key exists in both DataFrames.
- `on` specifies the join key column(s)
- `how` defines the join type (`"inner"`, `"left"`, `"right"`, `"full"`, ...)


In [4]:
inner_df = df_1.join(df_2, on="CustomerID", how="inner")
inner_df.show()

+----------+---------+--------+-------+-----+-------+---------+-------------+----------+----------+-----------+------------------+-------------+--------+-----+--------------------+
|CustomerID|FirstName|LastName|Country|Score|OrderID|ProductID|SalesPersonID| OrderDate|  ShipDate|OrderStatus|       ShipAddress|  BillAddress|Quantity|Sales|        CreationTime|
+----------+---------+--------+-------+-----+-------+---------+-------------+----------+----------+-----------+------------------+-------------+--------+-----+--------------------+
|         2|    Kevin|   Brown|    USA|  900|      1|      101|            3|2025-01-01|2025-01-05|  Delivered|9833 Mt. Dias Blv.|1226 Shoe St.|       1|   10|2025-01-01 12:34:...|
|         3|     Mary|    NULL|    USA|  750|      2|      102|            3|2025-01-05|2025-01-10|    Shipped|    250 Race Court|         NULL|       1|   15|2025-01-05 23:22:...|
|         1|   Jossef|Goldberg|Germany|  350|      3|      101|            5|2025-01-10|2025-01

**Note:** Both DataFrames must have a matching common column name for the join key.
If not, rename one of the datasets first.
It works like SQL, but you will get an error later when selecting the common column, because Spark doesn't know which of the two matching columns to use.


### 3.2 Other Join Types (left, right, full)

We can do left, right or full join by just changing the value of the `how` parameter:
- **Left join** - all rows from left, matching rows from right
- **Right join** - all rows from right, matching rows from left
- **Full join** - all rows from both DataFrames

**Workarounds:**
- For a right join, swap `df_1` and `df_2` and do a left join
- For a full join, do a left join + left-anti join, then union the results


### 3.3 Broadcast Join

When joining a large dataset with a small one (e.g., 500GB with 1GB), a normal shuffle join can fail with an out-of-memory (OOM) error.
`broadcast()` keeps the small DataFrame in memory and sends a full copy to every worker node, avoiding the expensive shuffle.


In [5]:
from pyspark.sql.functions import broadcast

broadcast_df = df_1.join(broadcast(df_2), on = "CustomerID", how="inner")
broadcast_df.show()

+----------+---------+--------+-------+-----+-------+---------+-------------+----------+----------+-----------+------------------+-------------+--------+-----+--------------------+
|CustomerID|FirstName|LastName|Country|Score|OrderID|ProductID|SalesPersonID| OrderDate|  ShipDate|OrderStatus|       ShipAddress|  BillAddress|Quantity|Sales|        CreationTime|
+----------+---------+--------+-------+-----+-------+---------+-------------+----------+----------+-----------+------------------+-------------+--------+-----+--------------------+
|         1|   Jossef|Goldberg|Germany|  350|      7|      102|            1|2025-02-15|2025-02-27|  Delivered|  136 Balboa Court|         NULL|       2|   30|2025-02-16 06:22:...|
|         1|   Jossef|Goldberg|Germany|  350|      4|      105|            3|2025-01-20|2025-01-25|    Shipped| 5724 Victory Lane|         NULL|       2|   60|2025-01-20 05:50:...|
|         1|   Jossef|Goldberg|Germany|  350|      3|      101|            5|2025-01-10|2025-01

## 4. Unions

`unionByName()` combines two DataFrames by column name and fills missing columns with NULL.
- `union()` requires the same number of columns in the same order
- `unionByName(allowMissingColumns=True)` handles different columns gracefully
- We can also use `unionAll()` if both DataFrames have the same number of columns
- `union()` behaves like `unionAll()` (keeps duplicates), so use `.distinct()` to remove duplicate rows


In [9]:
union_df = df_1.unionByName(df_2, allowMissingColumns=True)
union_df.show()

+----------+---------+--------+-------+-----+-------+---------+-------------+----------+----------+-----------+------------------+-------------+--------+-----+--------------------+
|CustomerID|FirstName|LastName|Country|Score|OrderID|ProductID|SalesPersonID| OrderDate|  ShipDate|OrderStatus|       ShipAddress|  BillAddress|Quantity|Sales|        CreationTime|
+----------+---------+--------+-------+-----+-------+---------+-------------+----------+----------+-----------+------------------+-------------+--------+-----+--------------------+
|         1|   Jossef|Goldberg|Germany|  350|   NULL|     NULL|         NULL|      NULL|      NULL|       NULL|              NULL|         NULL|    NULL| NULL|                NULL|
|         2|    Kevin|   Brown|    USA|  900|   NULL|     NULL|         NULL|      NULL|      NULL|       NULL|              NULL|         NULL|    NULL| NULL|                NULL|
|         3|     Mary|    NULL|    USA|  750|   NULL|     NULL|         NULL|      NULL|      N

## 5. Stop Spark Session

Always stop the session to release cluster resources.


In [10]:
spark.stop()